# Workflow visualiser

Shows the per-post LangGraph workflow and runs one sample post through it.

- **Simulation only:** the sample uses the `Simulator` publisher; nothing is sent to LinkedIn.
- **Isolated:** the sample run uses a fresh temporary folder and database, never your real `data/` or `database/app.db`.
- Install the optional tools once: `python -m pip install -e ".[dev,notebook]"`.
- Commit this notebook **with outputs cleared** (a test enforces it).

## 1. Safety check: LangSmith tracing must be off

In [ ]:
import os

for var in ("LANGSMITH_TRACING", "LANGCHAIN_TRACING_V2"):
    if os.environ.get(var, "").strip().lower() in ("true", "1"):
        raise RuntimeError(f"{var} is on. Turn it off: tracing would send post data to an external service.")
print("LangSmith tracing is off.")

## 2. The graph as a picture

`draw_mermaid_png()` renders through the **mermaid.ink** web service, which receives only the graph structure
(node names and edges), never post content or paths. If it fails (for example offline), paste the Mermaid
text from the next section into https://mermaid.live instead.

In [ ]:
from IPython.display import Image, display

from social_media_auto_poster.graph.workflow import build_workflow

graph = build_workflow().get_graph()
try:
    display(Image(graph.draw_mermaid_png()))
except Exception as error:
    print(f"Could not render the image ({type(error).__name__}: {error}).")
    print("Paste the Mermaid text below into https://mermaid.live to see the diagram.")

## 3. Mermaid text (paste into docs or mermaid.live)

In [ ]:
print(graph.draw_mermaid())

## 4. Run one sample post

Creates a valid post (a real 1x1 JPEG and a caption) in a temporary sandbox using the helpers from
`scripts/app.py`, runs it through `run_job` with the simulator, and shows the final `JobState`.

In [ ]:
import importlib.util
import tempfile
from pathlib import Path
from pprint import pprint

from social_media_auto_poster.config import Settings
from social_media_auto_poster.graph.state import WorkflowContext
from social_media_auto_poster.graph.workflow import run_job
from social_media_auto_poster.ingestion.scanner import scan_incoming
from social_media_auto_poster.models import Platform, PublishMode
from social_media_auto_poster.platforms.simulator import Simulator
from social_media_auto_poster.storage.database import open_app_db
from social_media_auto_poster.storage.jobs import JobRepository

# Reuse the sample-file helpers from scripts/app.py (no duplicated logic).
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
spec = importlib.util.spec_from_file_location("dev_app", repo_root / "scripts" / "app.py")
dev_app = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dev_app)

sandbox = Path(tempfile.mkdtemp(prefix="autoposter-notebook-"))
settings = Settings(base_dir=sandbox, data_dir=sandbox / "data")
incoming = settings.incoming_dir(Platform.LINKEDIN)
incoming.mkdir(parents=True)
dev_app.add_valid_post(incoming, "product-launch", "Our new product is live!")

[candidate] = scan_incoming(incoming, Platform.LINKEDIN).candidates
conn = open_app_db(settings.app_db_path)
try:
    context = WorkflowContext(
        publisher=Simulator(),
        mode=PublishMode.SIMULATED,
        repository=JobRepository(conn),
        settings=settings,
        stability_interval_s=0,
    )
    final_state = run_job(candidate, context)
    record = JobRepository(conn).get(final_state["job_id"])
finally:
    conn.close()

print(f"Sandbox: {sandbox}\n")
pprint(final_state, sort_dicts=False)
print(f"\nDatabase row: status={record.status}, mode={record.mode}")